# Padronização, Preparação e Validação da Landing Zone

## Objetivo
Criar catálogo, schemas e landing zone, e validar a presença e integridade dos arquivos de entrada antes da ingestão.

**Convenções deste projeto:**
- Comentários e textos: português (PT-BR)
- Identificadores (variáveis, funções): inglês (EN)
  - Constantes: `UPPER_SNAKE_CASE`
  - Demais variáveis: `lower_snake_case`
- Colunas e nomes de tabela: português, `snake_case`
- Símbolos nos logs: ✅ ⚠️ ❌

In [0]:
import re
from datetime import datetime

# Padronização dos nomes dos schemas e volumes
CATALOG = "movies_medallion"
BRONZE_SCHEMA_NAME = "bronze"
SILVER_SCHEMA_NAME = "silver"
GOLD_SCHEMA_NAME = "gold"

BRONZE_SCHEMA = f"{CATALOG}.{BRONZE_SCHEMA_NAME}"
SILVER_SCHEMA = f"{CATALOG}.{SILVER_SCHEMA_NAME}"
GOLD_SCHEMA = f"{CATALOG}.{GOLD_SCHEMA_NAME}"

LANDING_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA_NAME}/landing"

# Símbolos padronizados de log
OK = "✅"
FAIL = "❌"
WARN = "⚠️"

# Para verificação visual
print(f"CATALOG: {CATALOG}")
print(f"BRONZE_SCHEMA: {BRONZE_SCHEMA}")
print(f"SILVER_SCHEMA: {SILVER_SCHEMA}")
print(f"GOLD_SCHEMA: {GOLD_SCHEMA}")
print(f"LANDING_PATH: {LANDING_PATH}")

CATALOG: movies_medallion
BRONZE_SCHEMA: movies_medallion.bronze
SILVER_SCHEMA: movies_medallion.silver
GOLD_SCHEMA: movies_medallion.gold
LANDING_PATH: /Volumes/movies_medallion/bronze/landing


## Criação do Catalog, Schemas e Volume
Cria a estrutura base do projeto usando `IF NOT EXISTS`.

- Idempotente: rodar 2x não quebra nem duplica

In [0]:
# Cria, apenas se já não existem os schemas e volumes garantindo idempotência.
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {BRONZE_SCHEMA}.landing")

DataFrame[]

## Resumo Final - Validação da Landing Zone
Apenas reporta. Validações por bloco já deram raise antes.



In [0]:
# Utiliza o módulo re (regex) para corrigir possíveis títulos errados antes de verificar a presença do arquivo.
# Cria o dicionário contendo os nomes corretos e os regex para encontrar os nomes errados.
expected_files = {
    "movies_info_IMDB_TMDB.csv": r"movies_info_.*\.csv$",
    "movies_financials_IMDB_TMDB.csv": r"movies_financials_.*\.csv$",
    "movies_metrics_IMDB_TMDB.csv": r"movies_metrics_.*\.csv$",
    "credits_and_tags_IMDB_TMDB.csv": r"credits_and_tags_.*\.csv$",
    "movies_reviews.csv": r"movies_reviews.*\.csv$",
    "cotacao_dolar.json": r"cotacao_dolar.*\.json$",
}

# Verifica se todos os arquivos estão presentes e corrige o nome se necessário, removendo datas, por exemplo.
file_infos = dbutils.fs.ls(LANDING_PATH)
found_files = [f.name for f in file_infos]

empty_files = [f.name for f in file_infos if f.size == 0]
if empty_files:
    print(f"{FAIL} Arquivos vazios detectados: {empty_files}")
    raise Exception(f"Arquivos vazios encontrados: {empty_files}")

print("Encontrados:", found_files)

renamed_files = []
missing_files = []


for right_name, regex in expected_files.items():
    if right_name in found_files:
        continue

    wrong_name = None
    for i in found_files:
        if re.match(regex, i):
            wrong_name = i
            break

    if wrong_name:
        print(f"Renomeando {wrong_name} -> {right_name}")
        dbutils.fs.mv(f"{LANDING_PATH}/{wrong_name}", f"{LANDING_PATH}/{right_name}")
        found_files.remove(wrong_name)
        found_files.append(right_name)
        renamed_files.append((wrong_name, right_name))
    else:
        missing_files.append(right_name)

if missing_files:
    print(f"{FAIL} Arquivos faltando detectados")
    raise Exception(f"Arquivos faltando: {missing_files}")
else:
    print(f"{OK} Todos os arquivos estão presentes.")

Encontrados: ['cotacao_dolar.json', 'credits_and_tags_IMDB_TMDB.csv', 'movies_financials_IMDB_TMDB.csv', 'movies_info_IMDB_TMDB.csv', 'movies_metrics_IMDB_TMDB.csv', 'movies_reviews.csv']
✅ Todos os arquivos estão presentes.


In [0]:
# Resumo final: apenas reporta o estado após as validações acima.

# Arquivos não esperados na landing.
extra_files = [f for f in found_files if f not in expected_files]

print("=" * 60)
print("RESUMO DA VALIDAÇÃO DA LANDING")
print("=" * 60)

print(f"{OK} Arquivos válidos: {len(found_files)}")

if renamed_files:
    print(f"{WARN} Arquivos renomeados ({len(renamed_files)}):")
    for old, new in renamed_files:
        print(f"    - {old} → {new}")
else:
    print(f"{OK} Nenhum arquivo precisou ser renomeado")

if extra_files:
    print(f"{WARN} Arquivos não esperados ({len(extra_files)}):")
    for f in extra_files:
        print(f"    - {f}")
else:
    print(f"{OK} Nenhum arquivo inesperado na landing")

print(f"Executado em: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

RESUMO DA VALIDAÇÃO DA LANDING
✅ Arquivos válidos: 6
✅ Nenhum arquivo precisou ser renomeado
✅ Nenhum arquivo inesperado na landing
Executado em: 2026-10-16 17:41:37
